# 🎮 Ćwiczenie: Feature Engineering — Statystyki e-sportowe

**Kurs:** Python w Zastosowaniach Big Data (Studia stacjonarne)
**Zajęcia nr:** 7
**Szacowany czas:** ⏱️ ~90 minut
**Poziom:** ⭐⭐ Średni

## 📚 Cele nauczania

Po ukończeniu tego notebooka będziesz potrafić:
1. Uzupełniać brakujące wartości numeryczne i kategoryczne (`SimpleImputer`)
2. Kodować zmienne kategoryczne — nominalne (`OneHotEncoder`) i ordinalne (`OrdinalEncoder`)
3. Tworzyć nowe cechy (ratio, binning) i wykrywać outliery
4. Budować pełny `Pipeline` + `ColumnTransformer` bez wycieku danych (data leakage)

## 📋 Wymagania wstępne
- Podstawy pandas (DataFrame, filtrowanie, `groupby`)
- Wykład 7: Feature Engineering

---
> 💡 **Wskazówka:** Uruchamiaj komórki `Shift+Enter`. Jeśli utkniesz, przeczytaj komentarze `# TODO` — mówią dokładnie, co zrobić!

In [ ]:
# @title 🔧 Setup — uruchom tę komórkę jako pierwszą!

# !pip install -q scikit-learn pandas numpy

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 50)

np.random.seed(42)

print("✅ Setup zakończony! Do dzieła 🚀")

## 📖 Szybkie przypomnienie

| Pojęcie | Opis | Przykład |
|---------|------|----------|
| `SimpleImputer` | Uzupełnia braki (mediana/moda) | `SimpleImputer(strategy="median")` |
| `OneHotEncoder` | Kodowanie kategorii bez porządku | `country` → kolumny 0/1 |
| `OrdinalEncoder` | Kodowanie kategorii z porządkiem | `rank`: Brąz(0) < ... < Wyzwanie(6) |
| `StandardScaler` | Skalowanie mean=0, std=1 | `avg_reaction_ms` |
| `ColumnTransformer` | Różne transformacje dla różnych kolumn | numeryczne + kategoryczne razem |
| `Pipeline` | Łańcuch kroków: preprocessing → model | powtarzalność, brak wycieku danych |

In [ ]:
# @title 📊 Generowanie danych — statystyki graczy e-sportowych

n_players = 5000
rank_order = ["Brąz", "Srebro", "Złoto", "Platyna", "Diament", "Mistrz", "Wyzwanie"]
countries = ["PL", "DE", "KR", "US", "FR", "BR"]

# Bazowa "umiejętność" gracza (0-1) determinuje resztę statystyk z szumem
skill = np.clip(np.random.beta(2, 3, n_players), 0, 1)

matches_played = np.random.randint(10, 3000, n_players)
hours_per_week = np.clip(3 + skill * 35 + np.random.randn(n_players) * 4, 0.5, 60).round(1)

# Wyższa umiejętność -> wyższy win_rate i kda, z szumem
win_rate = np.clip(0.25 + skill * 0.5 + np.random.randn(n_players) * 0.05, 0.05, 0.95).round(3)
kda = np.clip(0.8 + skill * 4 + np.random.randn(n_players) * 0.4, 0.1, 8.0).round(2)

# Wyższa umiejętność -> niższy (lepszy) czas reakcji
avg_reaction_ms = np.clip(320 - skill * 150 + np.random.randn(n_players) * 20, 120, 400).round(0)

# Ranga zależna od skill (kwantyle) + trochę losowości
rank_index = np.clip(
    (skill * len(rank_order) + np.random.randn(n_players) * 0.6).astype(int),
    0, len(rank_order) - 1,
)
rank = np.array(rank_order)[rank_index]

country = np.random.choice(countries, n_players, p=[0.35, 0.15, 0.15, 0.15, 0.1, 0.1])

# Target: wygrana najbliższego meczu (1) czy porażka (0) - zależna od win_rate
won_next_match = (np.random.rand(n_players) < win_rate).astype(int)

df = pd.DataFrame({
    "player_id": range(1, n_players + 1),
    "matches_played": matches_played,
    "win_rate": win_rate,
    "kda": kda,
    "avg_reaction_ms": avg_reaction_ms,
    "rank": rank,
    "hours_per_week": hours_per_week,
    "country": country,
    "won_next_match": won_next_match,
})

# Wprowadzamy braki danych
for col, frac in [("kda", 0.05), ("avg_reaction_ms", 0.04), ("rank", 0.03), ("country", 0.02)]:
    missing_idx = df.sample(frac=frac, random_state=42).index
    df.loc[missing_idx, col] = np.nan

# Wprowadzamy kilka outlierów
outlier_idx = df.sample(n=10, random_state=7).index
df.loc[outlier_idx, "avg_reaction_ms"] = np.random.uniform(800, 1200, len(outlier_idx))

print(f"📊 Kształt danych: {df.shape}")
print(f"📋 Kolumny: {list(df.columns)}")
df.head()

## 🎓 Guided example — `ColumnTransformer` + `Pipeline`

**Zadanie:** Zbudujmy podstawowy pipeline, który skaluje cechy numeryczne i koduje `country` (nominalna), zanim jeszcze zajmiemy się rangą i nowymi cechami.

In [ ]:
# @title 🎓 Guided Example — uruchom i przeanalizuj

numeric_features = ["matches_played", "win_rate", "kda", "avg_reaction_ms", "hours_per_week"]
categorical_features = ["country"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),   # 👈 mediana - odporna na outliery
    ("scaler", StandardScaler()),                     # 👈 mean=0, std=1
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

basic_preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features),
])

basic_pipeline = Pipeline(steps=[("preprocessor", basic_preprocessor)])

transformed_sample = basic_pipeline.fit_transform(df.head(5))
print(f"Kształt po transformacji: {transformed_sample.shape}")
print(transformed_sample.round(2))

---
## ✏️ Ćwiczenie 1: Uzupełnienie brakujących wartości numerycznych

**Trudność:** ⭐

Uzupełnij braki w kolumnie `kda` medianą kolumny (bez modyfikowania oryginalnego `df` — pracuj na kopii `df_ex1`).

**Oczekiwany wynik:** `df_ex1["kda"].isnull().sum() == 0`

In [ ]:
# ✏️ Ćwiczenie 1: Uzupełnienie braków w 'kda'

df_ex1 = df.copy()

# TODO: policz medianę kolumny 'kda' (pomijając NaN) i uzupełnij nią braki
kda_median = ...  # Zastąp ... swoim kodem
df_ex1["kda"] = df_ex1["kda"].fillna(kda_median)

print(f"Mediana kda: {kda_median}")
print(f"Braki po uzupełnieniu: {df_ex1['kda'].isnull().sum()}")

In [ ]:
# @title ✅ Test — Ćwiczenie 1

assert kda_median is not ..., "❌ Nie policzyłeś mediany kolumny 'kda'"
assert df_ex1["kda"].isnull().sum() == 0, "❌ W kolumnie 'kda' nadal są braki"
assert abs(kda_median - df["kda"].median()) < 1e-9, "❌ Mediana nie zgadza się z oczekiwaną wartością"
print("✅ Świetnie! Braki w 'kda' uzupełnione poprawnie! 🎉")

---
## ✏️ Ćwiczenie 2: Uzupełnienie brakujących wartości kategorycznych

**Trudność:** ⭐

Uzupełnij braki w kolumnie `country` najczęściej występującą wartością (modą), pracując na `df_ex2` (kopia `df_ex1`).

> 💡 **Podpowiedź:** `df["col"].mode()[0]` zwraca najczęstszą wartość.

In [ ]:
# ✏️ Ćwiczenie 2: Uzupełnienie braków w 'country'

df_ex2 = df_ex1.copy()

# TODO: znajdź najczęstszą wartość (modę) kolumny 'country' i uzupełnij nią braki
country_mode = ...  # Zastąp ... swoim kodem
df_ex2["country"] = df_ex2["country"].fillna(country_mode)

print(f"Moda 'country': {country_mode}")
print(f"Braki po uzupełnieniu: {df_ex2['country'].isnull().sum()}")

In [ ]:
# @title ✅ Test — Ćwiczenie 2

assert country_mode is not ..., "❌ Nie policzyłeś mody kolumny 'country'"
assert df_ex2["country"].isnull().sum() == 0, "❌ W kolumnie 'country' nadal są braki"
assert country_mode == df["country"].mode()[0], "❌ Moda nie zgadza się z oczekiwaną wartością"
print("✅ Braki w 'country' uzupełnione poprawnie! 🎉")

---
## ✏️ Ćwiczenie 3: Kodowanie rangi jako zmiennej ordinalnej

**Trudność:** ⭐⭐

Ranga gracza ma naturalny porządek: `Brąz < Srebro < Złoto < Platyna < Diament < Mistrz < Wyzwanie`.
Uzupełnij braki w `rank` modą, a następnie zakoduj kolumnę przy pomocy `OrdinalEncoder` z jawnie podaną kolejnością (`rank_order`, zdefiniowaną w komórce z generowaniem danych).

**Oczekiwany wynik:** nowa kolumna `rank_encoded` z wartościami 0-6.

In [ ]:
# ✏️ Ćwiczenie 3: OrdinalEncoder dla 'rank'

df_ex3 = df_ex2.copy()
df_ex3["rank"] = df_ex3["rank"].fillna(df_ex3["rank"].mode()[0])

# TODO: stwórz OrdinalEncoder z categories=[rank_order]
rank_encoder = ...  # Zastąp ... swoim kodem, np. OrdinalEncoder(categories=[rank_order])

# TODO: zastosuj fit_transform na kolumnie 'rank' (pamiętaj o .reshape(-1, 1) lub df[["rank"]])
df_ex3["rank_encoded"] = ...  # Zastąp ... swoim kodem

print(df_ex3[["rank", "rank_encoded"]].drop_duplicates().sort_values("rank_encoded"))

In [ ]:
# @title ✅ Test — Ćwiczenie 3

assert rank_encoder is not ..., "❌ Nie stworzyłeś OrdinalEncoder"
assert "rank_encoded" in df_ex3.columns, "❌ Brak kolumny 'rank_encoded'"
assert df_ex3["rank_encoded"].min() == 0, "❌ Najniższa ranga powinna mieć kod 0"
assert df_ex3["rank_encoded"].max() == 6, "❌ Najwyższa ranga powinna mieć kod 6"
bronze_code = df_ex3.loc[df_ex3["rank"] == "Brąz", "rank_encoded"].iloc[0]
challenger_code = df_ex3.loc[df_ex3["rank"] == "Wyzwanie", "rank_encoded"].iloc[0]
assert bronze_code < challenger_code, "❌ 'Brąz' powinien mieć niższy kod niż 'Wyzwanie'"
print("✅ Ranga zakodowana poprawnie z zachowaniem porządku! 🎉")

---
## ✏️ Ćwiczenie 4: Nowa cecha — `efficiency_score`

**Trudność:** ⭐⭐

Stwórz nową cechę `efficiency_score` łączącą `win_rate` i znormalizowane `kda` (podzielone przez maksimum kolumny):

$$\text{efficiency\_score} = 0.5 \cdot \text{win\_rate} + 0.5 \cdot \frac{\text{kda}}{\max(\text{kda})}$$

> 💡 **Podpowiedź:** Pracuj na `df_ex3` (ma już uzupełnione `kda` z Ćwiczenia 1 poprzez `df_ex2`).

In [ ]:
# ✏️ Ćwiczenie 4: efficiency_score

df_ex4 = df_ex3.copy()

# TODO: oblicz efficiency_score wg wzoru powyżej
df_ex4["efficiency_score"] = ...  # Zastąp ... swoim kodem

print(df_ex4[["win_rate", "kda", "efficiency_score"]].head())

In [ ]:
# @title ✅ Test — Ćwiczenie 4

assert "efficiency_score" in df_ex4.columns, "❌ Brak kolumny 'efficiency_score'"
expected = 0.5 * df_ex4["win_rate"] + 0.5 * (df_ex4["kda"] / df_ex4["kda"].max())
assert np.allclose(df_ex4["efficiency_score"], expected, atol=1e-6), "❌ Wzór 'efficiency_score' jest niepoprawny"
assert df_ex4["efficiency_score"].between(0, 1).all(), "❌ efficiency_score powinien mieścić się w [0, 1]"
print("✅ Cecha 'efficiency_score' stworzona poprawnie! 🎉")

---
## ✏️ Ćwiczenie 5: Binning godzin gry

**Trudność:** ⭐⭐

Podziel `hours_per_week` na 4 kategorie przy pomocy `pd.cut()`:
- `casual`: 0-5h
- `regularny`: 5-15h
- `hardcore`: 15-30h
- `no-life`: 30h+

Zapisz wynik w nowej kolumnie `hours_category`.

In [ ]:
# ✏️ Ćwiczenie 5: binning godzin gry

df_ex5 = df_ex4.copy()

# TODO: użyj pd.cut z bins=[0, 5, 15, 30, 100] i odpowiednimi labels
df_ex5["hours_category"] = ...  # Zastąp ... swoim kodem

print(df_ex5["hours_category"].value_counts())

In [ ]:
# @title ✅ Test — Ćwiczenie 5

assert "hours_category" in df_ex5.columns, "❌ Brak kolumny 'hours_category'"
expected_categories = {"casual", "regularny", "hardcore", "no-life"}
actual_categories = set(df_ex5["hours_category"].dropna().unique().astype(str))
assert actual_categories.issubset(expected_categories), f"❌ Nieoczekiwane kategorie: {actual_categories}"
assert df_ex5["hours_category"].isnull().sum() == 0, "❌ Niektóre wartości nie zostały przypisane do kategorii (sprawdź zakresy bins)"
print("✅ Binning godzin gry wykonany poprawnie! 🎉")

---
## ✏️ Ćwiczenie 6: Wykrywanie i obsługa outlierów

**Trudność:** ⭐⭐⭐

Kolumna `avg_reaction_ms` zawiera kilka nierealistycznych wartości (>800ms). Użyj metody **IQR** do wykrycia outlierów i usuń je z `df_ex6`.

**Kroki:**
1. Policz Q1, Q3 i IQR dla `avg_reaction_ms`
2. Wyznacz granice `lower` i `upper` (1.5 × IQR)
3. Odfiltruj wiersze poza granicami

In [ ]:
# ✏️ Ćwiczenie 6: outliery w avg_reaction_ms (metoda IQR)

df_ex6 = df_ex5.copy()
n_before = len(df_ex6)

# TODO: policz Q1, Q3, IQR dla 'avg_reaction_ms' (pomijając NaN)
Q1 = ...  # Zastąp ... swoim kodem
Q3 = ...  # Zastąp ... swoim kodem
IQR = Q3 - Q1

# TODO: wyznacz granice lower/upper
lower = ...  # Zastąp ... swoim kodem
upper = ...  # Zastąp ... swoim kodem

# TODO: odfiltruj wiersze mieszczące się w granicach (uwzględnij NaN - zostaw je!)
df_ex6 = df_ex6[
    df_ex6["avg_reaction_ms"].isnull() | df_ex6["avg_reaction_ms"].between(lower, upper)
]

print(f"Wiersze przed: {n_before}, po usunięciu outlierów: {len(df_ex6)}")

In [ ]:
# @title ✅ Test — Ćwiczenie 6

assert Q1 is not ... and Q3 is not ..., "❌ Nie policzyłeś Q1/Q3"
assert lower is not ... and upper is not ..., "❌ Nie wyznaczyłeś granic lower/upper"
assert len(df_ex6) < n_before, "❌ Żaden outlier nie został usunięty - sprawdź granice"
assert (df_ex6["avg_reaction_ms"].dropna() <= upper).all(), "❌ Nadal są wartości powyżej górnej granicy"
print(f"✅ Usunięto {n_before - len(df_ex6)} outlierów! 🎉")

---
## ✏️ Ćwiczenie 7: Poprawny train/test split (bez wycieku danych)

**Trudność:** ⭐⭐⭐

Zbuduj kompletny `ColumnTransformer` (imputacja + skalowanie numeryczne, OneHot dla `country`, Ordinal dla `rank`) i `Pipeline` z `RandomForestClassifier`, przewidujący `won_next_match`.

**Kluczowe zasady:**
- Podziel dane na train/test **przed** jakimkolwiek `fit()`
- Użyj `stratify=y`, aby zachować proporcje klas
- `pipeline.fit()` wywołaj **tylko** na `X_train`

In [ ]:
# ✏️ Ćwiczenie 7: train/test split + pełny Pipeline

df_full = df_ex6.copy()

feature_columns = [
    "matches_played", "win_rate", "kda", "avg_reaction_ms",
    "hours_per_week", "rank", "country",
]
X = df_full[feature_columns]
y = df_full["won_next_match"]

# TODO: podziel X, y na X_train, X_test, y_train, y_test
# Użyj test_size=0.2, random_state=42, stratify=y
X_train, X_test, y_train, y_test = ...  # Zastąp ... swoim kodem

numeric_features_full = ["matches_played", "win_rate", "kda", "avg_reaction_ms", "hours_per_week"]

numeric_transformer_full = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_transformer_full = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])
ordinal_transformer_full = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OrdinalEncoder(categories=[rank_order])),
])

full_preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer_full, numeric_features_full),
    ("cat", categorical_transformer_full, ["country"]),
    ("ord", ordinal_transformer_full, ["rank"]),
])

full_pipeline = Pipeline(steps=[
    ("preprocessor", full_preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=200, random_state=42)),
])

# TODO: wytrenuj pipeline TYLKO na danych treningowych
...  # Zastąp ... swoim kodem, np. full_pipeline.fit(X_train, y_train)

test_accuracy = full_pipeline.score(X_test, y_test)
print(f"📊 Accuracy na zbiorze testowym: {test_accuracy:.2%}")

In [ ]:
# @title ✅ Test — Ćwiczenie 7

assert len(X_train) + len(X_test) == len(X), "❌ Podział train/test nie sumuje się do całości"
assert abs(len(X_test) / len(X) - 0.2) < 0.02, "❌ test_size powinien wynosić ok. 0.2"
assert hasattr(full_pipeline.named_steps["classifier"], "classes_"), "❌ Pipeline nie został wytrenowany (fit)"
assert test_accuracy > 0.5, "❌ Model działa gorzej niż losowe zgadywanie - sprawdź dane wejściowe"
print(f"✅ Pipeline zbudowany i wytrenowany poprawnie! Accuracy: {test_accuracy:.2%} 🎉")

---
## 🏆 Bonus Challenge (opcjonalny)

Wykorzystaj wytrenowany `full_pipeline` z Ćwiczenia 7, aby wyznaczyć **ważność cech** (`feature_importances_`) z `RandomForestClassifier`. Które cechy najbardziej wpływają na przewidywanie wygranej?

> 💡 **Podpowiedź:** Nazwy cech po transformacji uzyskasz przez `full_pipeline.named_steps["preprocessor"].get_feature_names_out()`.

In [ ]:
# 🏆 Bonus: ważność cech

feature_names = full_pipeline.named_steps["preprocessor"].get_feature_names_out()
importances = full_pipeline.named_steps["classifier"].feature_importances_

importance_df = pd.DataFrame({
    "cecha": feature_names,
    "ważność": importances,
}).sort_values("ważność", ascending=False)

print("🔑 Ranking ważności cech:")
print(importance_df.to_string(index=False))

---
## 📝 Podsumowanie

Świetna robota! 🎉 W tym ćwiczeniu przećwiczyłeś/aś:

- ✅ Uzupełnianie brakujących wartości numerycznych i kategorycznych (`SimpleImputer`)
- ✅ Kodowanie ordinalne (`rank`) i nominalne (`country`)
- ✅ Tworzenie nowych cech: ratio (`efficiency_score`) i binning (`hours_category`)
- ✅ Wykrywanie i usuwanie outlierów metodą IQR
- ✅ Budowę pełnego `Pipeline` + `ColumnTransformer` z poprawnym train/test split (bez data leakage)

### 📚 Dalsze materiały
- [scikit-learn — Preprocessing data](https://scikit-learn.org/stable/modules/preprocessing.html)
- [scikit-learn — ColumnTransformer](https://scikit-learn.org/stable/modules/compose.html#columntransformer-for-heterogeneous-data)

### 🔮 Co dalej?
Na **Wykładzie 8 (24.11.2026)** wykorzystamy dziś przygotowane cechy do trenowania i ewaluacji prawdziwych modeli klasyfikacji i regresji!